# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a worked example for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the FAIR^2 dataset
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(url)

# Access and display the metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, and their associated fields (by `@id`).
In Croissant datasets, each tabular file (or resource) usually corresponds to a `RecordSet`. We will programmatically extract their identifiers for inspection.

In [ ]:
# List all record sets by @id and summarize their fields
record_sets = list(dataset.record_sets)

print(f"Number of record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"\nRecordSet Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', '(none)')}")
    print(f"  Available fields:")
    for field in rs.fields:
        print(f"    - {field.id} (name: {field.name}, type: {field.data_type})")

## 3. Data Extraction
Load the data from **each record set** into a pandas DataFrame for analysis. All table, column, and field references use their `@id`s for robustness.

In [ ]:
# Prepare a dictionary of DataFrames, one per record set
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # List of dict records for chosen record set
    records = list(dataset.records(record_set=record_set_id))
    # If there are records, save as DataFrame
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)

if not dataframes:
    print("No tabular data found in dataset.")
else:
    # Pick the first table for display
    sample_rs_id = list(dataframes.keys())[0]
    print(f"Fields (columns) for RecordSet {sample_rs_id}:\n  {dataframes[sample_rs_id].columns.tolist()}")
    dataframes[sample_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply standard data exploration: filter records using a numeric field, normalize, and group by another field.
You must use appropriate field and record set `@id`s.

In [ ]:
# Choose the main patient summary table (RecordSet)
main_rs_id = sample_rs_id  # Use the first loaded table by default
df = dataframes[main_rs_id]
print(f"Selected record set: {main_rs_id}")

# Show all column names with their @id for EDA selection
print("Available field (column) @ids:")
for i, col_id in enumerate(df.columns):
    print(f"  {i}: {col_id}")

# Choose a numeric field by inspecting the field names
# For demonstration, we'll look for typical numeric fields like 'age', 'Interval', or 'Number' in their names,
# otherwise pick the first field.
import re
numeric_field_id = None
for col in df.columns:
    if re.search(r'age|interval|number|score|count|size|Years?', str(col), re.I):
        numeric_field_id = col
        break
if numeric_field_id is None:
    numeric_field_id = df.columns[0]
print(f"\nUsing numeric field: {numeric_field_id}")

# Convert to float/int where possible
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].median() if df[numeric_field_id].notnull().any() else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (sample):")
print(filtered_df.head(3))

# Normalize chosen numeric field
if filtered_df[numeric_field_id].notnull().any():
    filtered_df[numeric_field_id + "_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() else 1)
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head(3))

# Grouping: choose a likely categorical/grouping field
group_field_id = None
for col in df.columns:
    if re.search(r'sex|gender|type|location|site|Group|category', str(col), re.I):
        group_field_id = col
        break
if group_field_id is not None and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame("mean_" + str(numeric_field_id))
    print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the numeric field and the group comparison (if possible).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
if numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.xlabel(str(numeric_field_id))
    plt.title(f"Distribution of {numeric_field_id} in {main_rs_id}")
    plt.show()

# Boxplot by group if grouping field present
if group_field_id is not None and group_field_id in df.columns:
    plt.figure(figsize=(9,5))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.xlabel(str(group_field_id))
    plt.ylabel(str(numeric_field_id))
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated exploring the FAIR^2 dataset using Croissant and `mlcroissant`. We retrieved record sets and fields using their `@id`s, loaded data into DataFrames, filtered by a numeric field, normalized, grouped by a categorical field, and visualized distributions.

Key steps and findings:
- The dataset contains rich clinical records with multiple fields (see overview section).
- Data can be accessed and referenced robustly using `@id` for each feature/column.
- Standard EDA and visualization steps apply once loaded into pandas DataFrames.

For further work, refer to additional Croissant schema `@id` entities for more granular field selection or combine multiple record sets as needed.